# 20. 74쪽 그림 설명 보완과 전체 저장 준비

**이번 단계:** 74쪽 그림 설명 두 개를 Python 처리에 반영하고, 전체 자료를 DB 행 형태로 미리 준비합니다.

- [프로젝트 적용] 기존 한국어 모델·768차원·128토큰 청킹·SQL Mapper를 유지합니다.
- [프로젝트 추가] 전체 처리는 새 작업 번호(run_id)를 사용합니다. 기존 PDF와 등록된 그림은 재사용합니다.
- 아래 실행 셀은 읽기 전용 미리보기입니다. 임베딩·DB 쓰기·그림 업로드를 하지 않습니다.
- DB 행은 테이블의 컬럼 이름에 맞춘 딕셔너리입니다. 아직 벡터를 계산하지 않은 청크의 embedding은 None입니다.
- 프로젝트 .venv 커널을 선택하고 위에서 아래로 실행합니다. 전체 PDF 준비에 시간이 걸리므로 중복 실행하지 마세요.
- 새 노트북의 실행 결과는 아직 저장하지 않았습니다.


In [ ]:
from pathlib import Path
import sys

# 실행 위치가 notebooks여도 프로젝트 루트를 찾아 개인 Python 코드를 불러옵니다.
project_folder = next((folder for folder in (Path.cwd(), *Path.cwd().parents)
                       if (folder / "src/car_search_rag/zzong_santafe_lag").is_dir()), None)
if project_folder is None:
    raise FileNotFoundError("프로젝트 폴더에서 이 노트북을 열어 주세요.")
sys.path.insert(0, str(project_folder / "src"))

from car_search_rag.zzong_santafe_lag.full_store_service import FullManualStoreService


## 1. 전체 행 미리보기

`prepare()`는 글과 청크를 만들고 기존 PDF·그림을 DB에서 읽어 같은 번호로 연결합니다. 기존 행을 수정하지 않습니다.

여기서 기존 자료 번호는 재사용하고 **새 작업 번호만** 발급합니다. 미리보기를 다시 만들면 새 UUID가 생기지만 글 내용 식별값은 입력이 같을 때 같아야 합니다. 전체 저장은 마지막 검토한 입력 식별값이 일치할 때만 진행하도록 구성했습니다.


In [ ]:
# [프로젝트 추가] prepare는 읽기 전용 조회와 메모리 행 변환만 실행합니다.
service = FullManualStoreService().prepare(progress=print)
info = service.summary()

print("전체 처리에서 참조할 행 수:", info["row_counts"])
print("새 작업 저장 시 추가할 행 수:", info["additional_rows_if_new_run"])
print("재사용할 그림:", info["reuse_images"])
print("기존 개인 DB:", info["db_snapshot"])
print("입력 내용 식별값:", info["input_manifest_sha256"])
print("임베딩 완료 여부:", info["embedding_complete"])
print("DB 저장 여부:", info["db_written"])


## 2. 74쪽 설명이 들어갔는지 보기

PDF 74쪽 위 그림은 다리 받침대 주의사항 앞에 있고, 아래 그림은 릴렉션 기능 제목 아래에 있습니다. 두 설명에 각각 이 문맥을 붙였습니다. 75쪽에는 릴렉션 설명과 주의사항이 이어집니다.

현재 부모는 앞좌석 전체(PDF 70~85쪽)를 묶은 자동 초안입니다. 두 그림 배치를 확인해도 전체 주제 경계·그림 대응을 모두 검토한 것은 아니므로 **auto_draft_needs_review**와 페이지 수준 연결 표시는 유지합니다.


In [ ]:
# [프로젝트 추가] 보존 원문(raw_text)은 그대로 두고 검색 글(content)에 설명을 추가했습니다.
for record in service.service.parents:
    if record["metadata"].get("image_description_updates"):
        print("부모:", record["metadata"]["record_id"], record["metadata"]["title"])
        print("검토 상태:", record["metadata"]["verification_status"])
        print("검색 글 끝부분:\n", record["content"][-400:])
        for part in record["image_parts"]:
            if part["pdf_page_number"] == 74:
                print("그림 이름:", part["name"], "| 관련 제목:", part["source_heading"])
                print("그림 설명:", part["description"])


## 3. 아직 계산하지 않은 벡터 확인

None은 아직 값이 없다는 뜻입니다. 임의의 0 벡터로 채워 저장하지 않습니다. 저장 코드는 벡터의 크기·숫자 유효성·정규화와 원문 연결을 확인한 뒤에만 ready 상태를 지정합니다.


In [ ]:
# 모델을 실행하지 않고 준비된 청크의 상태만 셉니다.
rows = service.preview.tables["chunks"]
print("전체 청크 수:", len(rows))
print("아직 임베딩이 없는 청크:", sum(row["embedding"] is None for row in rows))
print("가장 긴 청크 토큰:", max(row["token_count"] for row in rows))
print("설명 있는 그림 연결:", info["image_links_with_description"])
print("설명 없는 그림 연결:", info["image_links_without_description"])


## 실제 미리보기 결과와 다음 순서: 임베딩 → 저장

2026-10-02 full-preview 종료 코드 0: 부모 527개·청크 2,213개·고유 그림 864개·연결 1,157건. 설명 있는 연결 12건·없는 연결 1,145건입니다. 기존 개인 DB는 23행 그대로이며 벡터는 아직 None입니다.

입력 식별값: 893151d66a49fab8f21060ee95e7917054ebb12753dc62eb4ead26cd52798a1b. 원문·각주·그림 설명·출처·검토 상태·모델·청크를 함께 구분하는 값입니다. 코드나 자료를 바꾸면 새 미리보기를 확인합니다.

이 노트북에는 자동으로 실행되는 저장 셀이 없습니다. 현재 단계는 코드·미리보기 준비까지입니다.

1. **full-preview:** DB 읽기와 메모리 행 준비. 지금 실행한 단계입니다.
2. **full-embed:** 전체 청크를 로컬 모델로 계산하고 메모리에 연결. DB에 쓰지 않습니다.
3. **full-save:** 검토한 입력 식별값과 승인 옵션을 확인하고 임베딩 후 개인 DB에 저장.

터미널의 다음 계산 단계는 아래 명령입니다. 아직 이번 작업에서 실행하지 않았습니다.

```powershell
.\.venv\Scripts\python.exe src\car_search_rag\zzong_santafe_lag\manual.py full-embed
```

새 터미널 실행은 이전 실행의 메모리 벡터를 재사용하지 않으므로 full-save도 다시 계산합니다. 노트북에서 같은 service 객체를 유지하는 방식의 임베딩·저장 연결은 다음 단계에서 안내합니다. 저장에는 `--confirm-save`와 마지막 미리보기의 `--manifest-sha256`이 필요합니다.

저장 서비스는 새 run_id의 글·벡터·그림 참조를 한 트랜잭션으로 저장하고, 확정 전후 DB 값을 비교하도록 작성했습니다. 같은 완료 작업은 값·벡터를 대조해 재사용하고, 다른 값은 덮어쓰지 않습니다. 전체 embed/save·실패 복구·중복 재실행은 아직 실제 실행하지 않았습니다.

**코드 읽는 순서:** manual.py의 full-preview → full_store_service.py의 prepare/embed/save → mappers/manual_store.sql.txt.
